# Notebook 7 — IFRS 9 ECL: Full Quarterly Path Macro Scenario Analysis
## Mortgage Credit Risk Modelling

This notebook visualises the output of `07_macro_scenario_ifrs9.py` — the reimplemented macro scenario script that replaces the original single peak-quarter approximation with a proper IFRS 9 ECL calculation.

**Key improvements over the original script:**

| Dimension | Original `07_macro_scenario_analysis.py` | This script |
|---|---|---|
| Scoring | One prediction at peak quarter only | One prediction per loan **per quarter** |
| Macro path | 5 quarters, flat after shock vector ends | 20 quarters with explicit **recovery phase** |
| At-risk pool | Fixed — all loans scored every time | **Shrinks** as SP(q) = ∏(1 − PD(k)) accumulates |
| Loan ageing | `loan_age` frozen at observation date | Incremented **+3 months per quarter** |
| ECL formula | PD × LGD × EAD at peak | ∑ PD(q) × SP(q−1) × LGD × EAD × DF(q) |
| Horizons | Single (peak only) | **1Y / 2Y / 3Y / 5Y lifetime** |
| Recovery | UR stays elevated forever | UR mean-reverts toward long-run level |

**IFRS 9 ECL formula:**

$$\text{ECL}_i^s(H) = \sum_{q=1}^{H} \underbrace{PD_i^s(q)}_{\text{cond. default}} \cdot \underbrace{SP_i^s(q-1)}_{\text{survived to }q} \cdot LGD \cdot EAD_i \cdot \underbrace{\frac{1}{(1+r)^{q/4}}}_{\text{discount}}$$

**Prerequisites:** Run `01_data_preprocessing.py`, `03_pd_ensemble.py`, then `07_macro_scenario_ifrs9.py`.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

plt.rcParams.update({
    "figure.facecolor": "#0F1117", "axes.facecolor": "#0F1117",
    "axes.edgecolor":   "#2D3748", "axes.labelcolor": "#E2E8F0",
    "xtick.color":      "#A0AEC0", "ytick.color":     "#A0AEC0",
    "text.color":       "#E2E8F0", "grid.color":      "#1A2035",
    "legend.facecolor": "#1A2035", "legend.edgecolor":"#2D3748",
    "font.family": "monospace",    "figure.dpi": 120,
})

C = {
    "base":    "#10B981",   # green
    "adverse": "#F59E0B",   # amber
    "severe":  "#EF4444",   # red
    "weighted":"#3B82F6",   # blue
    "sky":     "#38BDF8",
    "muted":   "#6B7280",
}

SCEN_COLOR = {"Base": C["base"], "Adverse": C["adverse"], "Severe": C["severe"]}
SCEN_LABEL = {
    "Base":    "Base Scenario (60%)",
    "Adverse": "Adverse Scenario (30%)",
    "Severe":  "Severe Scenario / GFC-level (10%)",
}

PROC = Path("data/processed")
FIG  = Path("data/figures")

# ── Check outputs exist ────────────────────────────────────────────────────────
needed = [
    "ifrs9_ecl_by_loan.csv",
    "ifrs9_ecl_summary.csv",
    "ifrs9_macro_paths.csv",
]
missing = [f for f in needed if not (PROC / f).exists()]
if missing:
    print("⚠  Missing outputs — run 07_macro_scenario_ifrs9.py first:")
    for m in missing: print("   •", m)
else:
    ecl_loan    = pd.read_csv(PROC / "ifrs9_ecl_by_loan.csv")
    ecl_summary = pd.read_csv(PROC / "ifrs9_ecl_summary.csv")
    macro_paths = pd.read_csv(PROC / "ifrs9_macro_paths.csv")

    print(f"✓ ECL by loan:   {len(ecl_loan):,} loans,  {len(ecl_loan.columns)} columns")
    print(f"✓ ECL summary:   {len(ecl_summary)} rows")
    print(f"✓ Macro paths:   {len(macro_paths)} rows  ({macro_paths['quarter'].nunique()} quarters × {macro_paths['scenario'].nunique()} scenarios)")
    print()
    print("Horizons detected:", [c for c in ecl_loan.columns if c.startswith("ecl_base_")])
    print("Scenarios detected:", sorted(macro_paths["scenario"].unique()))


## 1. Macro Scenario Paths — Full 20-Quarter Projection

Each scenario now has three phases:

1. **Stress phase** — UR rises, HPI falls  
2. **Plateau phase** — peak stress held  
3. **Recovery phase** — UR mean-reverts (δₖ < 0), HPI partially recovers  

The original script had no recovery phase: δₖ = 0 for all k beyond the shock vector, meaning UR stayed permanently elevated forever. This overstated lifetime ECL significantly.


In [ ]:
if macro_paths is not None:
    fig, axes = plt.subplots(2, 2, figsize=(16, 9))
    fig.suptitle(
        "Macro Scenario Paths — 20-Quarter (5-Year) Projection with Recovery Phase\n"
        "Dashed vertical: Adverse peak (Q4) | Dotted vertical: Severe peak (Q6)",
        fontsize=12, fontweight="bold", color="white"
    )

    quarters = sorted(macro_paths["quarter"].unique())

    for scenario in ["Base", "Adverse", "Severe"]:
        sub = macro_paths[macro_paths["scenario"] == scenario].sort_values("quarter")
        col = SCEN_COLOR[scenario]
        lbl = SCEN_LABEL[scenario]

        # UR level
        axes[0,0].plot(sub["quarter"], sub["ur"],
                       color=col, linewidth=2.5, marker="o", markersize=3, label=lbl)
        # HPI change from origination
        axes[0,1].plot(sub["quarter"], (sub["hpi_ratio"] - 1) * 100,
                       color=col, linewidth=2.5, marker="o", markersize=3, label=lbl)
        # Quarterly UR increment
        ur_vals   = sub["ur"].values
        hpi_vals  = (sub["hpi_ratio"] - 1).values * 100
        ur_delta  = np.diff(ur_vals,  prepend=ur_vals[0])
        hpi_delta = np.diff(hpi_vals, prepend=hpi_vals[0])
        offset    = (["Base","Adverse","Severe"].index(scenario) - 1) * 0.22
        axes[1,0].bar(sub["quarter"] + offset, ur_delta,  width=0.22,
                       color=col, alpha=0.75, label=lbl)
        axes[1,1].bar(sub["quarter"] + offset, hpi_delta, width=0.22,
                       color=col, alpha=0.75, label=lbl)

    specs = [
        (axes[0,0], "Unemployment Rate Path", "UR (%)", "%.1f%%"),
        (axes[0,1], "HPI Change from Origination (%)", "HPI Change (%)", "%.1f%%"),
        (axes[1,0], "Quarterly UR Increment (δₖ)", "δ UR (pp)", "%+.2f"),
        (axes[1,1], "Quarterly HPI Increment (%)", "δ HPI (%)",  "%+.1f%%"),
    ]
    for ax, title, ylabel, fmt in specs:
        ax.set_title(title, color="#CBD5E1", fontsize=10)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_xlabel("Quarter", fontsize=9)
        ax.set_xticks(quarters[::2])
        ax.set_xticklabels([f"Q{q}" for q in quarters[::2]], fontsize=7)
        ax.legend(fontsize=7, loc="best")
        ax.grid(True, alpha=0.2)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.yaxis.set_major_formatter(mtick.FormatStrFormatter(fmt))
        # Mark peaks
        ax.axvline(4, color=C["adverse"], linewidth=0.8, linestyle="--", alpha=0.5)
        ax.axvline(6, color=C["severe"],  linewidth=0.8, linestyle=":",  alpha=0.5)

    axes[0,0].axhline(4.0, color=C["muted"], linewidth=1, linestyle=":", alpha=0.5,
                       label="Starting UR = 4%")
    axes[0,1].axhline(0, color=C["muted"], linewidth=1, linestyle="--")
    axes[1,0].axhline(0, color=C["muted"], linewidth=1, linestyle="--")
    axes[1,1].axhline(0, color=C["muted"], linewidth=1, linestyle="--")

    plt.tight_layout()
    plt.show()

    # Summary table
    print("Scenario path summary:")
    print("-" * 65)
    for scenario in ["Base", "Adverse", "Severe"]:
        sub = macro_paths[macro_paths["scenario"] == scenario]
        peak_q   = sub.loc[sub["ur"].idxmax(), "quarter"]
        peak_ur  = sub["ur"].max()
        final_ur = sub[sub["quarter"] == sub["quarter"].max()]["ur"].values[0]
        peak_hpi = sub["hpi_ratio"].min()
        deltas   = np.diff(sub.sort_values("quarter")["ur"].values)
        n_neg    = (deltas < 0).sum()
        print(f"  {scenario:8s}  peak UR={peak_ur:.1f}% at Q{peak_q}  "
              f"final UR={final_ur:.1f}%  "
              f"trough HPI={peak_hpi:.3f}  recovery quarters={n_neg}")


## 2. IFRS 9 ECL Summary — Probability-Weighted Provision at Four Horizons

Under IFRS 9:

- **Stage 1 loans** → 12-month ECL (1Y horizon)  
- **Stage 2 loans** → Lifetime ECL (full horizon)  
- **Stage 3 loans** → Credit-impaired, ECL = LGD × EAD  

The probability-weighted ECL is what appears on the balance sheet as a provision:

$$\text{ECL}_{\text{weighted}}(H) = \sum_s w_s \cdot \text{ECL}_s(H) = 0.60 \cdot \text{ECL}_{\text{base}} + 0.30 \cdot \text{ECL}_{\text{adverse}} + 0.10 \cdot \text{ECL}_{\text{severe}}$$


In [ ]:
if ecl_summary is not None:
    print("IFRS 9 ECL Summary — Full Quarterly Path Scoring")
    print("=" * 70)
    print(ecl_summary.to_string(index=False))
    print()

    # Detect horizon columns
    horizon_cols = [c for c in ecl_summary.columns if c.startswith("total_ecl_") and c.endswith("_$M")]
    horizons = [c.replace("total_ecl_","").replace("_$M","") for c in horizon_cols]

    if not horizon_cols:
        print("⚠  No ECL horizon columns found.")
    else:
        scen_rows = ecl_summary[~ecl_summary["scenario"].str.contains("WEIGHTED", na=False)].copy()
        weighted  = ecl_summary[ecl_summary["scenario"].str.contains("WEIGHTED", na=False)].copy()

        n_hor   = len(horizon_cols)
        n_scen  = len(scen_rows)
        width   = 0.20
        x       = np.arange(n_hor)

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle(
            "IFRS 9 ECL by Horizon and Scenario\n"
            "ECL = Σ PD(q) × SP(q−1) × LGD × EAD × DF(q)   "
            "with at-risk pool shrinkage and loan ageing",
            fontsize=11, fontweight="bold", color="white"
        )

        # Left: per-scenario ECL at each horizon
        scen_name_map = {
            "Base Scenario": "Base",
            "Adverse Scenario": "Adverse",
            "Severe Scenario (GFC-level)": "Severe",
        }
        for i, (_, row) in enumerate(scen_rows.iterrows()):
            scen_key = scen_name_map.get(row["scenario"], row["scenario"])
            col  = SCEN_COLOR.get(scen_key, C["sky"])
            lbl  = SCEN_LABEL.get(scen_key, row["scenario"])
            vals = [row.get(hc, 0) for hc in horizon_cols]
            offs = (i - n_scen/2 + 0.5) * width
            bars = axes[0].bar(x + offs, vals, width=width, color=col,
                               alpha=0.85, edgecolor="none", label=lbl)
            for bar, v in zip(bars, vals):
                axes[0].text(bar.get_x() + bar.get_width()/2,
                              bar.get_height() + 0.001,
                              f"${v:.3f}M", ha="center", va="bottom",
                              fontsize=7.5, color="#E2E8F0")

        axes[0].set_xticks(x)
        axes[0].set_xticklabels(horizons, fontsize=10)
        axes[0].set_ylabel("Total ECL ($M)", fontsize=10)
        axes[0].set_title("Scenario ECL by Horizon", color="#CBD5E1", fontsize=10)
        axes[0].legend(fontsize=8)
        axes[0].grid(True, axis="y", alpha=0.25)
        axes[0].spines["top"].set_visible(False)
        axes[0].spines["right"].set_visible(False)

        # Right: probability-weighted ECL
        if not weighted.empty:
            w_vals = [float(weighted[hc].values[0]) for hc in horizon_cols]
            bars = axes[1].bar(x, w_vals, width=0.5, color=C["weighted"], alpha=0.85)
            for bar, v in zip(bars, w_vals):
                axes[1].text(bar.get_x() + bar.get_width()/2,
                              bar.get_height() + 0.001,
                              f"${v:.4f}M", ha="center", va="bottom",
                              fontsize=9, color="white", fontweight="bold")

        axes[1].set_xticks(x)
        axes[1].set_xticklabels(horizons, fontsize=10)
        axes[1].set_ylabel("Probability-Weighted ECL ($M)", fontsize=10)
        axes[1].set_title(
            "Probability-Weighted ECL by Horizon\n"
            "Base 60% / Adverse 30% / Severe 10%",
            color="#CBD5E1", fontsize=10
        )
        axes[1].grid(True, axis="y", alpha=0.25)
        axes[1].spines["top"].set_visible(False)
        axes[1].spines["right"].set_visible(False)

        plt.tight_layout()
        plt.show()


## 3. Per-Loan ECL Distribution — Tail Risk by Scenario and Horizon

The per-loan ECL distribution shows how losses are concentrated. In a severe scenario, the right tail of the distribution expands dramatically — a small subset of high-risk loans (high CLTV, low FICO, delinquent) drives a disproportionate share of total ECL.

The comparison between the 1Y and lifetime horizons shows how much additional expected loss is captured by the full path integration vs the simple 12-month Stage 1 estimate.


In [ ]:
if ecl_loan is not None:
    # Detect ECL columns: ecl_{scenario}_{horizon}
    ecl_cols = [c for c in ecl_loan.columns if c.startswith("ecl_")]
    horizons_found = sorted(set(c.rsplit("_", 1)[-1] for c in ecl_cols))
    scenarios_found = sorted(set(c.split("_")[1] for c in ecl_cols))

    # Pick first and last horizon for comparison
    h_first = horizons_found[0]   # e.g. 1Y
    h_last  = horizons_found[-1]  # e.g. 5Y / lifetime

    fig, axes = plt.subplots(2, 3, figsize=(17, 9))
    fig.suptitle(
        f"Per-Loan ECL Distribution — {h_first} vs {h_last} Horizon\n"
        "Showing concentration of loss in the tail across scenarios",
        fontsize=12, fontweight="bold", color="white"
    )

    for col_idx, scenario in enumerate(["base", "adverse", "severe"]):
        col_first = f"ecl_{scenario}_{h_first}"
        col_last  = f"ecl_{scenario}_{h_last}"
        scen_key  = scenario.capitalize()
        color     = SCEN_COLOR.get(scen_key, C["sky"])

        for row_idx, (col, h_label) in enumerate([
            (col_first, h_first), (col_last, h_last)
        ]):
            ax = axes[row_idx, col_idx]
            if col not in ecl_loan.columns:
                ax.text(0.5, 0.5, f"Column\n{col}\nnot found",
                        ha="center", va="center", transform=ax.transAxes, color=C["muted"])
                continue

            vals = ecl_loan[col].dropna()
            vals_k = vals / 1000   # in $K for readability

            ax.hist(vals_k, bins=60, color=color, alpha=0.75,
                    edgecolor="none", density=True)

            p50 = np.percentile(vals_k, 50)
            p95 = np.percentile(vals_k, 95)
            p99 = np.percentile(vals_k, 99)

            ax.axvline(p95, color="white", linewidth=1.2, linestyle="--",
                        label=f"p95=${p95:.1f}K")
            ax.axvline(p99, color="#FBBF24", linewidth=1.2, linestyle=":",
                        label=f"p99=${p99:.1f}K")

            ax.set_title(
                f"{SCEN_LABEL.get(scen_key, scen_key)} — {h_label}\n"
                f"Mean=${vals_k.mean():.2f}K  Total=${vals.sum()/1e6:.3f}M",
                color="#CBD5E1", fontsize=8.5
            )
            ax.set_xlabel("ECL per Loan ($K)", fontsize=8)
            ax.set_ylabel("Density", fontsize=8)
            ax.legend(fontsize=7.5)
            ax.grid(True, alpha=0.2)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.show()

    # Summary statistics
    print("\nPer-Loan ECL Statistics by Scenario and Horizon ($K):")
    print("-" * 80)
    for h in horizons_found:
        print(f"\nHorizon: {h}")
        for scenario in ["base", "adverse", "severe"]:
            col = f"ecl_{scenario}_{h}"
            if col not in ecl_loan.columns:
                continue
            v = ecl_loan[col].dropna() / 1000
            print(f"  {scenario.capitalize():8s}  mean={v.mean():.3f}K  "
                  f"p50={v.quantile(.5):.3f}K  p95={v.quantile(.95):.3f}K  "
                  f"p99={v.quantile(.99):.3f}K  total=${v.sum()/1e3:.3f}M")


## 4. At-Risk Pool Shrinkage and Survival Curves

A key feature of the proper IFRS 9 computation is that the at-risk pool shrinks each quarter as loans default and are removed:

$$SP_i(q) = \prod_{k=1}^{q} (1 - PD_i(k))$$

This means that by Q20, a non-trivial fraction of high-risk loans have already defaulted and contribute zero ECL to later quarters. The original peak-only approach ignored this entirely — it implicitly assumed every loan was at full risk in every period.

The ECL contribution per quarter also shows the peak loss quarter, which is typically **not** the macro stress peak but slightly after it, as the survival pool catches up.


In [ ]:
if ecl_loan is not None:
    horizons_found = sorted(set(c.rsplit("_", 1)[-1]
                                 for c in ecl_loan.columns if c.startswith("ecl_")))
    scenarios_found = ["base", "adverse", "severe"]

    # Reconstruct incremental ECL per quarter from cumulative horizon ECLs
    # We approximate the quarterly ECL contribution as the difference between
    # consecutive horizon ECLs when the horizons are 1Y apart.
    # For a full decomposition, the raw PD and SP matrices from the script
    # would need to be saved — this uses what is available in the outputs.

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(
        "ECL Accumulation Over Horizon — Quarterly Path Integration\n"
        "Illustrates how lifetime ECL builds vs 1Y Stage 1 estimate",
        fontsize=12, fontweight="bold", color="white"
    )

    # Left: cumulative portfolio ECL by horizon
    for scenario in scenarios_found:
        scen_key = scenario.capitalize()
        color    = SCEN_COLOR.get(scen_key, C["sky"])
        cumulative_ecls = []
        for h in horizons_found:
            col = f"ecl_{scenario}_{h}"
            if col in ecl_loan.columns:
                cumulative_ecls.append((h, ecl_loan[col].sum() / 1e6))

        if cumulative_ecls:
            hs, vals = zip(*cumulative_ecls)
            axes[0].plot(range(len(hs)), vals, color=color, linewidth=2.5,
                          marker="o", markersize=8,
                          label=SCEN_LABEL.get(scen_key, scen_key))
            axes[0].fill_between(range(len(hs)), 0, vals,
                                  color=color, alpha=0.08)
            for xi, (h, v) in enumerate(zip(hs, vals)):
                axes[0].annotate(f"${v:.3f}M", (xi, v),
                                  textcoords="offset points", xytext=(0, 8),
                                  ha="center", fontsize=8, color=color)

    axes[0].set_xticks(range(len(horizons_found)))
    axes[0].set_xticklabels(horizons_found, fontsize=10)
    axes[0].set_ylabel("Total Portfolio ECL ($M)", fontsize=10)
    axes[0].set_title("Cumulative ECL by Horizon", color="#CBD5E1", fontsize=10)
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.2)
    axes[0].spines["top"].set_visible(False)
    axes[0].spines["right"].set_visible(False)

    # Right: ECL uplift ratio — lifetime vs 1Y
    h_short = horizons_found[0]
    h_long  = horizons_found[-1]
    ratios  = []
    labels  = []
    colors  = []
    for scenario in scenarios_found:
        col_s = f"ecl_{scenario}_{h_short}"
        col_l = f"ecl_{scenario}_{h_long}"
        if col_s in ecl_loan.columns and col_l in ecl_loan.columns:
            ecl_s = ecl_loan[col_s].sum()
            ecl_l = ecl_loan[col_l].sum()
            ratio = ecl_l / max(ecl_s, 1e-9)
            ratios.append(ratio)
            labels.append(f"{scenario.capitalize()}\n({h_short}→{h_long})")
            colors.append(SCEN_COLOR.get(scenario.capitalize(), C["sky"]))

    bars = axes[1].bar(range(len(ratios)), ratios, color=colors,
                        alpha=0.85, edgecolor="none", width=0.5)
    for bar, r in zip(bars, ratios):
        axes[1].text(bar.get_x() + bar.get_width()/2,
                      bar.get_height() + 0.02,
                      f"×{r:.2f}", ha="center", va="bottom",
                      fontsize=12, color="white", fontweight="bold")

    axes[1].axhline(1.0, color=C["muted"], linewidth=1.5, linestyle="--",
                     label="1× = no additional loss beyond 1Y")
    axes[1].set_xticks(range(len(ratios)))
    axes[1].set_xticklabels(labels, fontsize=9)
    axes[1].set_ylabel(f"ECL Ratio: {h_long} / {h_short}", fontsize=10)
    axes[1].set_title(
        f"Lifetime vs 1Y ECL Ratio\n"
        f"How much additional loss is captured beyond the 12-month Stage 1 estimate",
        color="#CBD5E1", fontsize=10
    )
    axes[1].legend(fontsize=9)
    axes[1].grid(True, axis="y", alpha=0.25)
    axes[1].spines["top"].set_visible(False)
    axes[1].spines["right"].set_visible(False)

    plt.tight_layout()
    plt.show()


## 5. Loan-Level ECL: Cross-Scenario Comparison

For each loan, the ECL under the severe scenario exceeds the base scenario ECL. The scatter plot below shows this relationship — loans far above the diagonal are those most sensitive to macro stress (typically: high CLTV, low FICO, already delinquent, high unemployment exposure).

This is the loan-level input to IFRS 9 Stage 2 assessment: loans where the severe scenario ECL is materially above the base ECL are candidates for transfer to Stage 2 (lifetime ECL recognition).


In [ ]:
if ecl_loan is not None:
    horizons_found = sorted(set(c.rsplit("_", 1)[-1]
                                 for c in ecl_loan.columns if c.startswith("ecl_")))
    h = horizons_found[0]   # use 1Y horizon for this comparison

    col_base   = f"ecl_base_{h}"
    col_adv    = f"ecl_adverse_{h}"
    col_sev    = f"ecl_severe_{h}"

    if all(c in ecl_loan.columns for c in [col_base, col_adv, col_sev]):
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle(
            f"Loan-Level ECL: Cross-Scenario Comparison ({h} Horizon)\n"
            "Points above diagonal: loan is more sensitive to macro stress",
            fontsize=12, fontweight="bold", color="white"
        )

        base_k = ecl_loan[col_base] * 1000    # in $
        adv_k  = ecl_loan[col_adv]  * 1000
        sev_k  = ecl_loan[col_sev]  * 1000

        # Adverse vs Base
        axes[0].scatter(base_k, adv_k, s=4, alpha=0.3, color=C["adverse"],
                         linewidths=0, rasterized=True)
        m = max(base_k.max(), adv_k.max())
        axes[0].plot([0, m], [0, m], "k--", linewidth=1.2, alpha=0.6, label="Equal ECL")
        axes[0].set_xlabel(f"Base ECL ($)", fontsize=10)
        axes[0].set_ylabel(f"Adverse ECL ($)", fontsize=10)
        axes[0].set_title("Adverse vs Base", color="#CBD5E1", fontsize=10)
        axes[0].legend(fontsize=9)
        axes[0].grid(True, alpha=0.15)
        axes[0].spines["top"].set_visible(False)
        axes[0].spines["right"].set_visible(False)

        # Severe vs Base
        axes[1].scatter(base_k, sev_k, s=4, alpha=0.3, color=C["severe"],
                         linewidths=0, rasterized=True)
        m = max(base_k.max(), sev_k.max())
        axes[1].plot([0, m], [0, m], "k--", linewidth=1.2, alpha=0.6, label="Equal ECL")
        axes[1].set_xlabel(f"Base ECL ($)", fontsize=10)
        axes[1].set_ylabel(f"Severe ECL ($)", fontsize=10)
        axes[1].set_title("Severe vs Base", color="#CBD5E1", fontsize=10)
        axes[1].legend(fontsize=9)
        axes[1].grid(True, alpha=0.15)
        axes[1].spines["top"].set_visible(False)
        axes[1].spines["right"].set_visible(False)

        plt.tight_layout()
        plt.show()

        # Uplift statistics
        uplift_adv = (adv_k / base_k.clip(lower=1e-9))
        uplift_sev = (sev_k / base_k.clip(lower=1e-9))
        print(f"\nECL uplift ratio at {h} horizon (scenario / base):")
        print(f"  Adverse:  mean={uplift_adv.mean():.2f}×  "
              f"p50={uplift_adv.quantile(.5):.2f}×  "
              f"p95={uplift_adv.quantile(.95):.2f}×")
        print(f"  Severe:   mean={uplift_sev.mean():.2f}×  "
              f"p50={uplift_sev.quantile(.5):.2f}×  "
              f"p95={uplift_sev.quantile(.95):.2f}×")

        # Top 10 most stressed loans
        stress_df = ecl_loan[["loan_seq_num", col_base, col_sev]].copy()                     if "loan_seq_num" in ecl_loan.columns                     else ecl_loan[[col_base, col_sev]].copy()
        stress_df["sev_uplift"] = sev_k.values / base_k.clip(lower=1e-9).values
        stress_df = stress_df.sort_values("sev_uplift", ascending=False).head(10)
        print(f"\nTop 10 loans by severe/base ECL uplift ratio:")
        print(stress_df.to_string(index=False))


## 6. Original vs Improved ECL: What Changes?

The original script computed ECL as:

$$\text{ECL}_i^{\text{original}} = PD_i(\text{peak quarter}) \times LGD \times EAD_i$$

The new script computes:

$$\text{ECL}_i^{\text{new}}(H) = \sum_{q=1}^{H} PD_i(q) \cdot SP_i(q-1) \cdot LGD \cdot EAD_i \cdot DF(q)$$

The differences arise from four sources:
1. **Path integration** — averaging over all quarters, not just the worst
2. **At-risk pool shrinkage** — SP(q−1) < 1 for q > 1
3. **Recovery phase** — PD declines after the stress peak
4. **Discounting** — future losses are worth less than immediate losses

This cell reconstructs a proxy for the original peak-quarter ECL for comparison.


In [ ]:
if ecl_loan is not None and macro_paths is not None:
    horizons_found = sorted(set(c.rsplit("_", 1)[-1]
                                 for c in ecl_loan.columns if c.startswith("ecl_")))
    h_1y = horizons_found[0]

    LGD = 0.40
    DISCOUNT_R = 0.05

    comparison_rows = []
    for scenario in ["base", "adverse", "severe"]:
        scen_key = scenario.capitalize()
        col_new = f"ecl_{scenario}_{h_1y}"
        if col_new not in ecl_loan.columns:
            continue

        # New ECL (from script)
        ecl_new_total = ecl_loan[col_new].sum() / 1e6

        # Proxy for original: peak UR quarter, score everyone once (no SP, no discount)
        # We use the original_upb from ecl_loan as EAD
        ead = ecl_loan["orig_upb"].values if "orig_upb" in ecl_loan.columns               else np.full(len(ecl_loan), 200_000.0)

        # Find peak quarter UR for this scenario
        scen_macro = macro_paths[macro_paths["scenario"] == scen_key]
        peak_ur = scen_macro["ur"].max()

        # We do not have the exact peak-quarter PD in the output files,
        # but we can derive a proxy:
        # proxy ECL_original = (ECL_1Y_new / first-quarter discount)
        #                      * (1 / SP assumption) approximately
        # Better: use the 1Y ECL divided by 4 as the single-quarter equivalent
        # and then multiply by 1 (no survival, no discount) as rough proxy.
        # Annotate clearly that this is an approximation.
        ecl_1q_proxy = ecl_new_total * 4   # rough: annualise from quarterly

        comparison_rows.append({
            "Scenario": scen_key,
            "New ECL 1Y ($M)": round(ecl_new_total, 4),
            "~Original ECL proxy ($M)": round(ecl_1q_proxy, 4),
            "New / Original ratio": round(ecl_new_total / max(ecl_1q_proxy, 1e-9), 3),
        })

    comp_df = pd.DataFrame(comparison_rows)
    print("ECL Comparison: Full Path vs Peak-Quarter Proxy")
    print("Note: Original proxy is approximate — requires the raw PD matrix to be exact.")
    print("=" * 65)
    print(comp_df.to_string(index=False))
    print()

    if not comp_df.empty:
        fig, ax = plt.subplots(figsize=(10, 5))
        fig.suptitle(
            "New IFRS 9 ECL vs Original Peak-Quarter Approximation (1Y Horizon)\n"
            "Note: Original figure is a proxy — see cell notes",
            fontsize=11, fontweight="bold", color="white"
        )

        x = np.arange(len(comp_df))
        w = 0.35
        bars1 = ax.bar(x - w/2, comp_df["New ECL 1Y ($M)"],     width=w,
                        color=C["weighted"], alpha=0.85, label="New: Full quarterly path")
        bars2 = ax.bar(x + w/2, comp_df["~Original ECL proxy ($M)"], width=w,
                        color=C["muted"],    alpha=0.85, label="~Original: Peak-quarter proxy")

        for bars in [bars1, bars2]:
            for bar in bars:
                h_bar = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2,
                         h_bar + 0.001,
                         f"${h_bar:.3f}M", ha="center", va="bottom",
                         fontsize=9, color="#E2E8F0")

        ax.set_xticks(x)
        ax.set_xticklabels(comp_df["Scenario"], fontsize=10)
        ax.set_ylabel("Total Portfolio ECL ($M)", fontsize=10)
        ax.legend(fontsize=9)
        ax.grid(True, axis="y", alpha=0.25)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        plt.tight_layout()
        plt.show()


## 7. Macro Path Sensitivity — Recovery Speed Impact

The shape of the recovery path affects lifetime ECL independently of the stress peak level. This section illustrates the contribution of the recovery phase: by how much does the ECL change if UR recovers faster vs slower after the peak?

This is a perturbation analysis — we compare the ECL at each horizon under the actual recovery path vs a counterfactual where UR stays at peak level forever (the original script's implicit assumption).


In [ ]:
if macro_paths is not None and ecl_summary is not None:
    horizons_found = sorted(set(c.rsplit("_", 1)[-1]
                                 for c in ecl_summary.columns
                                 if c.startswith("total_ecl_") and c.endswith("_$M")))

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.suptitle(
        "ECL at Each Horizon — Effect of Recovery Phase\n"
        "Without recovery: UR stays at peak forever (original script assumption)",
        fontsize=11, fontweight="bold", color="white"
    )

    scen_rows = ecl_summary[~ecl_summary["scenario"].str.contains("WEIGHTED", na=False)]

    scen_name_map = {
        "Base Scenario":               "Base",
        "Adverse Scenario":            "Adverse",
        "Severe Scenario (GFC-level)": "Severe",
    }

    x = np.arange(len(horizons_found))
    for i, (_, row) in enumerate(scen_rows.iterrows()):
        scen_key = scen_name_map.get(row["scenario"], row["scenario"])
        color    = SCEN_COLOR.get(scen_key, C["sky"])
        vals     = [row.get(f"total_ecl_{h}_$M", np.nan) for h in horizons_found]

        ax.plot(x, vals, color=color, linewidth=2.5, marker="o", markersize=7,
                 label=f"{scen_key} (with recovery)")

        # Counterfactual: if ECL grows linearly (no recovery, no SP shrinkage)
        # proxy: take 1Y ECL and extrapolate linearly to each horizon
        # (this approximates the original peak-only script's implicit assumption)
        if vals[0] and not np.isnan(vals[0]):
            h_nums = [int(h.replace("Y","")) if "Y" in h else 1 for h in horizons_found]
            counterfactual = [vals[0] * h_n for h_n in h_nums]
            ax.plot(x, counterfactual, color=color, linewidth=1.5,
                     linestyle="--", alpha=0.5,
                     label=f"{scen_key} (no recovery — original assumption)")

    ax.set_xticks(x)
    ax.set_xticklabels(horizons_found, fontsize=10)
    ax.set_ylabel("Total Portfolio ECL ($M)", fontsize=10)
    ax.set_xlabel("Horizon", fontsize=10)
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(True, alpha=0.2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()

    # Quantify the recovery benefit
    print("\nECL reduction from recovery phase (new vs no-recovery counterfactual):")
    print("-" * 65)
    for _, row in scen_rows.iterrows():
        scen_key = scen_name_map.get(row["scenario"], row["scenario"])
        v_1y = row.get(f"total_ecl_{horizons_found[0]}_$M", np.nan)
        v_lt = row.get(f"total_ecl_{horizons_found[-1]}_$M", np.nan)
        if np.isnan(v_1y) or np.isnan(v_lt):
            continue
        h_nums = [int(h.replace("Y","")) if "Y" in h else 1 for h in horizons_found]
        cf_lt  = v_1y * h_nums[-1]
        saving = cf_lt - v_lt
        print(f"  {scen_key:8s}  lifetime ECL with recovery: ${v_lt:.4f}M  "
              f"without: ${cf_lt:.4f}M  saving: ${saving:.4f}M")


## Summary

| Improvement | Original `07_macro_scenario_analysis.py` | This notebook's script |
|---|---|---|
| Scoring frequency | Once (peak quarter) | Every quarter × 20 quarters |
| Macro path | 5Q, flat after shock | 20Q with recovery |
| Recovery phase | None (UR stuck at peak) | Explicit δₖ < 0 quarters |
| At-risk pool | Fixed | SP(q) = ∏(1−PD(k)) shrinks |
| Loan ageing | Frozen | +3 months per quarter |
| Discounting | None | DF(q) = 1/(1+r)^(q/4) |
| Horizons | Peak only | 1Y / 2Y / 3Y / Lifetime |
| IFRS 9 compliance | Approximation | Closer to standard |

**Regulatory outputs produced:**

| Output | IFRS 9 / Basel purpose |
|---|---|
| `ifrs9_ecl_by_loan.csv` | Per-loan ECL input for Stage 1/2 assessment |
| `ifrs9_ecl_summary.csv` | Portfolio provision, Tier 1 capital impact |
| `ifrs9_macro_paths.csv` | Scenario documentation for model governance |
| ECL by horizon charts | Forward-looking information §5.5.17 |
| Lifetime vs 1Y ratio | Stage 1 → Stage 2 transfer trigger evidence |

**Remaining simplifications** (for a fully production-grade implementation):

- LGD is hardcoded at 40% — should use per-loan output from `04_lgd_models.py`  
- EAD uses `orig_upb` — should use current UPB adjusted for amortisation  
- Loan features other than `loan_age` do not evolve — requires a satellite transition model  
- PD model is not recalibrated per horizon — calibration from script 08 should be applied  
